### 1. Setup Inicial
 

In [11]:
# Importação das bibliotecas

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configurações de exibição

pd.set_option('display.max_columns', None)   
pd.set_option('display.float_format', '{:.2f}'.format)

print('Bibliotecas carregadas!')


Bibliotecas carregadas!


In [12]:
# Carregamento dos dados brutos

df_raw = pd.read_csv('../data/Base Varejo.csv', sep=';')

print('--- Dataframe bruto ---')
print(f'   Linhas x Colunas : {df_raw.shape[0]} x {df_raw.shape[1]}')
print(f'   Nulos totais     : {df_raw.isnull().sum().sum()}')
print(f'   Duplicatas       : {df_raw.duplicated().sum()}')

# Primeiras visualizações 

print('\n--- Informações do Dataframe ---')
display(df_raw.info())

print('\n--- Primeiras linhas do Dataframe ---')
display(df_raw.head(10))


--- Dataframe bruto ---
   Linhas x Colunas : 830000 x 14
   Nulos totais     : 3320000
   Duplicatas       : 96553

--- Informações do Dataframe ---
<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  str    
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  str    
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  str    
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  str    
 9   PR_NOME      830000 non-null  str    
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), str(5)
memory usage: 88.7 MB


None


--- Primeiras linhas do Dataframe ---


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN
5,01/02/2019,1000,534,M,4,1,C,187,HIGIENE,HASTES FLEXIVEIS,NaN,NaN,NaN,NaN
6,01/02/2019,1000,534,M,4,1,C,163,ALIMENTOS,MORTADELA,NaN,NaN,NaN,NaN
7,01/02/2019,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,NaN,NaN,NaN,NaN
8,01/02/2019,1000,534,M,4,1,C,95,LIMPEZA,AMACIANTE,NaN,NaN,NaN,NaN
9,01/02/2019,1000,534,M,4,1,C,198,BEBIDAS,ENERGETICO,NaN,NaN,NaN,NaN


#### Insigts

- A base está estruturada de forma transacional, pois existem valores nas colunas de ID de compra (`CO_ID`) e ID do cliente (`CL_ID`) que se repetem, tendo variação no ID do produto (`PR_ID`). Isso demostra uma jornada de compra.
- As 4 últimas colunas vieram vazias e os valores nulos encontrados percencem a elas. Eliminação necessária;
- Coluna de DATA precisa ser alterada para o formato `datetime`;
- Colunas de nome e categoria do produto precisam ser normalizadas;
- Coluna de gênero com abreviação;

### 2. Transformações

In [13]:
# Primeiro, copiar o dataframe para as transformações
df_limpo = df_raw.copy()

# Eliminar as últimas quatro colunas que estão totalmente vazias
df_limpo = df_limpo.dropna(how='all', axis=1)

# Converter a coluna 'DATA' para datetime
df_limpo['DATA'] = pd.to_datetime(df_limpo['DATA'], format='%d/%m/%Y')

# Visualização das mudanças
print('\n--- Dataframe após tratamentos iniciais ---')
display(df_limpo.head())


--- Dataframe após tratamentos iniciais ---


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,2019-02-01,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,2019-02-01,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,2019-02-01,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,2019-02-01,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
4,2019-02-01,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO


In [14]:
# Renomear os nomes das colunas para melhor visualização
df_limpo = df_limpo.rename(columns={
    'DATA': 'data_venda',
    'CO_ID': 'id_cupom',
    'CL_ID': 'id_cliente',
    'CL_GENERO': 'genero_cliente',
    'CL_EC': 'estado_civil_cliente',
    'CL_FHL': 'faixa_filhos_cliente',
    'CL_SEG': 'segmentacao_cliente',
    'PR_ID': 'id_produto',
    'PR_CAT': 'categoria_produto',
    'PR_NOME': 'nome_produto'
})

# Renomear os valores das colunas 'genero_cliente'
df_limpo['genero_cliente'] = df_limpo['genero_cliente'].replace({'M': 'Masculino', 'F': 'Feminino'})
print('--- Valores da coluna de gênero após o tratamento ---')
print(df_limpo['genero_cliente'].unique())

# Normalizar as colunas de nome e categoria dos produtos
df_limpo['nome_produto'] = df_limpo['nome_produto'].str.title()
df_limpo['categoria_produto'] = df_limpo['categoria_produto'].str.title() 

print('\n--- Dataframe após renomeações ---')
display(df_limpo.head(10))

--- Valores da coluna de gênero após o tratamento ---
<StringArray>
['Masculino', 'Feminino']
Length: 2, dtype: str

--- Dataframe após renomeações ---


,data_venda,id_cupom,id_cliente,genero_cliente,estado_civil_cliente,faixa_filhos_cliente,segmentacao_cliente,id_produto,categoria_produto,nome_produto
0,2019-02-01,1000,534,Masculino,4,1,C,67,Bebidas,Refrigerante Guarana
1,2019-02-01,1000,534,Masculino,4,1,C,70,Bebidas,Refrigerante Outros
2,2019-02-01,1000,534,Masculino,4,1,C,178,Higiene,Lenco Umedecido
3,2019-02-01,1000,534,Masculino,4,1,C,4,Alimentos,Abacaxi
4,2019-02-01,1000,534,Masculino,4,1,C,175,Limpeza,Limpador Multiuso
5,2019-02-01,1000,534,Masculino,4,1,C,187,Higiene,Hastes Flexiveis
6,2019-02-01,1000,534,Masculino,4,1,C,163,Alimentos,Mortadela
7,2019-02-01,1000,534,Masculino,4,1,C,11,Alimentos,Azeite
8,2019-02-01,1000,534,Masculino,4,1,C,95,Limpeza,Amaciante
9,2019-02-01,1000,534,Masculino,4,1,C,198,Bebidas,Energetico


### 3. Limpeza de Nulos e Duplicatas

In [15]:
# Verificação de nulos por colunas
nulos = df_limpo.isnull().sum()
pct = (nulos / len(df_limpo) * 100).round(1)

print(f'\n--- Nulos por coluna (%) ---')
print(pct)


--- Nulos por coluna (%) ---
data_venda             0.00
id_cupom               0.00
id_cliente             0.00
genero_cliente         0.00
estado_civil_cliente   0.00
faixa_filhos_cliente   0.00
segmentacao_cliente    0.00
id_produto             0.00
categoria_produto      0.00
nome_produto           0.00
dtype: float64


In [16]:
# Verificação de duplicatas por coluna

print(f'\nDuplicatas: {df_limpo.duplicated().sum()}')

#Verificas se as duplicadas são reais ou 
df_limpo[df_limpo.duplicated(keep=False)].head(10)


Duplicatas: 96553


,data_venda,id_cupom,id_cliente,genero_cliente,estado_civil_cliente,faixa_filhos_cliente,segmentacao_cliente,id_produto,categoria_produto,nome_produto
3,2019-02-01,1000,534,Masculino,4,1,C,4,Alimentos,Abacaxi
7,2019-02-01,1000,534,Masculino,4,1,C,11,Alimentos,Azeite
14,2019-02-01,1000,534,Masculino,4,1,C,13,Alimentos,Banana
15,2019-02-01,1000,534,Masculino,4,1,C,218,Alimentos,Bife De Coxao Mole
19,2019-02-01,1000,534,Masculino,4,1,C,13,Alimentos,Banana
22,2019-02-01,1000,534,Masculino,4,1,C,69,Bebidas,Refrigerante Limao
34,2019-02-01,1000,534,Masculino,4,1,C,225,Alimentos,Atum
40,2019-02-01,1000,534,Masculino,4,1,C,4,Alimentos,Abacaxi
46,2019-02-01,1000,534,Masculino,4,1,C,11,Alimentos,Azeite
49,2019-02-01,1000,534,Masculino,4,1,C,69,Bebidas,Refrigerante Limao


#### Insigths

- Ao analisar as duplicadas, conclui que não são duplicatas reais, mas sim o registo do mesmo item para o mesmo cliente no mesmo cupom. Com isso, decidi manter todos os valores e analisar melhor agrupando na etapa de estatística descritiva

 

### 4. Estatística Descritiva

In [17]:
# Visão descritiva geral
print('--- Análise descritiva da coluna de Número de filhos do cliente ---')
display(df_limpo['faixa_filhos_cliente'].describe())

print(f"Moda da coluna de filhos: {df_limpo['faixa_filhos_cliente'].mode()[0]}")

--- Análise descritiva da coluna de Número de filhos do cliente ---


count   830000.00
mean         1.15
std          1.42
min          0.00
25%          0.00
50%          0.00
75%          2.00
max          4.00
Name: faixa_filhos_cliente, dtype: float64

Moda da coluna de filhos: 0


### 5. Exploração

In [18]:
# Produto que mais vende
df_limpo['nome_produto'].value_counts().head(20)

nome_produto
Presunto Cozido         14381
Sardinha                 7490
Gel                      7399
Banana                   7385
Desengordurante          7378
Modelador                7363
Bife De Coxao Mole       7355
Removedor                7350
Escova De Dente          7346
Papinha Infantil         7346
Detergente               7345
Salgadinhos De Milho     7345
Coracao De Frango        7342
Refrigerante Outros      7338
Cera                     7336
Ricota                   7334
Rodo                     7328
Cebola                   7321
Refrigerante Limao       7320
Limpador Perfumado       7316
Name: count, dtype: int64

In [19]:
# Média de filhos por gênero
filhos_por_genero = df_limpo.groupby('genero_cliente')['faixa_filhos_cliente'].mean().reset_index(name='media_filhos')
print('--- Média de filhos por gênero dos clientes ---')
display(filhos_por_genero)

--- Média de filhos por gênero dos clientes ---


,genero_cliente,media_filhos
0,Feminino,1.09
1,Masculino,1.21


In [20]:
# Agrupamento de compras por cupom
compras = df_limpo.groupby('id_cupom').size().reset_index(name='quantidade_de_produtos') 
print('--- Quantidade de produtos por cupom ---')
display(compras.head(10))

print(f'Total de cupons = {compras.shape[0]}')

--- Quantidade de produtos por cupom ---


,id_cupom,quantidade_de_produtos
0,1000,52
1,1040,15
2,1078,82
3,1082,71
4,1103,6
5,1121,64
6,1188,78
7,1200,81
8,1240,37
9,1293,8


Total de cupons = 18471


In [21]:
# Itens comprados por segmento de cliente
compras_segmento = df_limpo.groupby('segmentacao_cliente').size().reset_index(name='total_itens_comprados')
print('--- Quantidade de produtos por segmento ---')

compras_segmento['percentual'] = (compras_segmento['total_itens_comprados'] / compras_segmento['total_itens_comprados'].sum()) * 100

display(compras_segmento)

--- Quantidade de produtos por segmento ---


,segmentacao_cliente,total_itens_comprados,percentual
0,A,67736,8.16
1,B,530163,63.88
2,C,232101,27.96
